# Complete ML Pipeline: Dataset to Azure ACI Deployment

This notebook covers:
1. Loading and preparing a dataset
2. Training a machine learning model
3. Deploying the model to Azure Container Instance (ACI)

## Prerequisites
- Azure subscription
- Azure Machine Learning workspace (you mentioned you already created this)
- Required Python packages (installed below)

## Step 1: Install Required Packages

In [ ]:
# Install required packages
!pip install azureml-core azureml-sdk scikit-learn pandas numpy joblib

## Step 2: Import Libraries

In [5]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import json

# Azure ML imports
from azureml.core import Workspace, Model, Environment
from azureml.core.webservice import AciWebservice, Webservice
from azureml.core.model import InferenceConfig
from azureml.core.conda_dependencies import CondaDependencies

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 3: Connect to Azure ML Workspace

**Important:** You need to provide your Azure ML workspace details here.

You can find these in the Azure Portal under your Machine Learning resource:
- Go to your ML workspace in Azure Portal
- Look for "Overview" section
- Note down: Subscription ID, Resource Group, and Workspace Name

In [11]:
# Option 1: Connect using config file (recommended)
# If you have a config.json file from Azure ML Studio, use this:
try:
    ws = Workspace.from_config()
    print(f"Connected to workspace: {ws.name}")
except:
    print("Config file not found. Please use Option 2 below.")
    ws = None

c:\Users\mahmo\anaconda3\envs\envML\lib\site-packages\msal\application.py:207: UserWarning: Please upgrade msal-extensions. Only msal-extensions 1.2+ can work with msal 1.30+
  "Please upgrade msal-extensions. "


Performing interactive authentication. Please follow the instructions on the terminal.


The default web browser has been opened at https://login.microsoftonline.com/organizations/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.


Config file not found. Please use Option 2 below.


In [12]:
# Option 2: Connect manually (use this if config file doesn't exist)
# Uncomment and fill in your details:

SUBSCRIPTION_ID = '747a4063-5f65-4bf4-9547-0460675fd69b'
RESOURCE_GROUP = 'MAHMOUD.MAFTAH-rg'
WORKSPACE_NAME = 'ML-ressource'

# ws = Workspace(subscription_id=subscription_id,
#                resource_group=resource_group,
#                workspace_name=workspace_name)

# print(f"Connected to workspace: {ws.name}")

In [ ]:
from azureml.core import Workspace
from azureml.core.authentication import InteractiveLoginAuthentication


auth = InteractiveLoginAuthentication(tenant_id="common")

ws = Workspace.get(
    name=WORKSPACE_NAME,
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
    auth=auth
)

print('e')

ws.write_config(path=".", file_name="configg.json")
print("✓ config.json created!")

## Step 4: Load and Prepare Dataset

We'll use the diabetes dataset as an example. You can replace this with your own dataset.

In [6]:
# Load dataset
diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = diabetes.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print("\nFirst few rows:")
print(X.head())

Dataset shape: (442, 10)
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

First few rows:
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005671 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019908 -0.017646  
1 -0.039493 -0.068330 -0.092204  
2 -0.002592  0.002864 -0.025930  
3  0.034309  0.022692 -0.009362  
4 -0.002592 -0.031991 -0.046641  


## Step 5: Split Data into Training and Testing Sets

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 353
Test set size: 89


## Step 6: Train Machine Learning Model

In [8]:
# Train Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.2f}")

Model Performance:
Mean Squared Error: 2945.29
R² Score: 0.44


## Step 7: Save the Model Locally

In [9]:
# Create model directory
import os
os.makedirs('model', exist_ok=True)

# Save model
model_path = 'model/diabetes_model.pkl'
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to model/diabetes_model.pkl


## Step 8: Create Scoring Script

This script will be used by Azure to handle inference requests.

In [10]:
%%writefile score.py
import json
import numpy as np
import joblib
from azureml.core.model import Model

def init():
    """Initialize the model"""
    global model
    # Get the path to the registered model
    model_path = Model.get_model_path('diabetes_model')
    model = joblib.load(model_path)
    print("Model loaded successfully")

def run(raw_data):
    """Make predictions on input data"""
    try:
        # Parse input data
        data = json.loads(raw_data)['data']
        
        # Convert to numpy array
        data = np.array(data)
        
        # Make prediction
        predictions = model.predict(data)
        
        # Return results
        return json.dumps({"predictions": predictions.tolist()})
    except Exception as e:
        error = str(e)
        return json.dumps({"error": error})

Overwriting score.py


## Step 9: Register Model in Azure ML Workspace

In [ ]:
# Register the model
model = Model.register(
    workspace=ws,
    model_name='diabetes_model',
    model_path=model_path,
    description='Random Forest model for diabetes prediction',
    tags={'type': 'regression', 'framework': 'sklearn'}
)

print(f"Model registered: {model.name}, Version: {model.version}")

## Step 10: Create Environment Configuration

In [ ]:
# Create environment
env = Environment(name='diabetes-env')

# Specify conda dependencies
conda_dep = CondaDependencies()
conda_dep.add_pip_package('azureml-defaults')
conda_dep.add_pip_package('scikit-learn==1.3.0')
conda_dep.add_pip_package('joblib')
conda_dep.add_pip_package('numpy')

env.python.conda_dependencies = conda_dep

print("Environment configuration created")

## Step 11: Configure Inference

In [ ]:
# Create inference configuration
inference_config = InferenceConfig(
    entry_script='score.py',
    environment=env
)

print("Inference configuration created")

## Step 12: Configure Azure Container Instance (ACI) Deployment

In [ ]:
# Configure ACI deployment
aci_config = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    auth_enabled=True,  # Enable key-based authentication
    description='Diabetes prediction model deployment',
    tags={'model': 'diabetes', 'framework': 'sklearn'}
)

print("ACI deployment configuration created")

## Step 13: Deploy Model to Azure Container Instance

⚠️ **This step may take 5-10 minutes to complete.**

In [ ]:
# Deploy the model
service_name = 'diabetes-service'

# Check if service already exists and delete it
try:
    existing_service = Webservice(workspace=ws, name=service_name)
    print(f"Service {service_name} already exists. Deleting it...")
    existing_service.delete()
    print("Existing service deleted.")
except:
    print(f"No existing service named {service_name}. Proceeding with deployment.")

# Deploy new service
print("Starting deployment... This may take several minutes.")
service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[model],
    inference_config=inference_config,
    deployment_config=aci_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)
print(f"\n✅ Deployment complete!")
print(f"Service state: {service.state}")
print(f"Scoring URI: {service.scoring_uri}")

## Step 14: Get Service Details and Authentication Key

In [ ]:
# Get authentication keys
primary_key, secondary_key = service.get_keys()

print("=" * 60)
print("DEPLOYMENT DETAILS")
print("=" * 60)
print(f"Service Name: {service.name}")
print(f"Scoring URI: {service.scoring_uri}")
print(f"\nPrimary Key: {primary_key}")
print(f"Secondary Key: {secondary_key}")
print("=" * 60)
print("\n⚠️  IMPORTANT: Save these keys securely!")

## Step 15: Test the Deployed Service

In [ ]:
import requests

# Prepare test data (using first row from test set)
test_sample = X_test.iloc[0:1].values.tolist()

# Create input data
input_data = json.dumps({
    'data': test_sample
})

# Set headers with authentication
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {primary_key}'
}

# Make prediction request
response = requests.post(service.scoring_uri, data=input_data, headers=headers)

print("Test Input:")
print(test_sample)
print(f"\nActual Value: {y_test.iloc[0]:.2f}")
print(f"\nAPI Response:")
print(response.json())

## Step 16: Example Code for Making Predictions from Any Python Script

In [ ]:
# This code can be used in any Python script to call your deployed model

example_code = f'''
import requests
import json

# Your deployment details
scoring_uri = "{service.scoring_uri}"
api_key = "{primary_key}"  # Use your primary or secondary key

# Prepare your input data (replace with your actual data)
# Data should match the format expected by your model
input_data = {{
    'data': [
        [0.03807591, 0.05068012, 0.06169621, 0.02187235, -0.0442235,
         -0.03482076, -0.04340085, -0.00259226, 0.01990842, -0.01764613]
    ]
}}

# Convert to JSON
data = json.dumps(input_data)

# Set headers
headers = {{
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {{api_key}}'
}}

# Make prediction request
response = requests.post(scoring_uri, data=data, headers=headers)

# Get prediction
prediction = response.json()
print("Prediction:", prediction)
'''

print("Copy this code to use in your applications:")
print("=" * 60)
print(example_code)

## Step 17: Monitor and Manage Your Deployment

In [ ]:
# Get service logs (useful for debugging)
logs = service.get_logs()
print("Service Logs:")
print(logs[:1000])  # Print first 1000 characters

## Optional: Clean Up Resources

Run this cell only when you want to delete the deployment to avoid charges.

In [ ]:
# Uncomment to delete the service
# service.delete()
# print("Service deleted successfully")

## Summary

✅ **What we've accomplished:**

1. Loaded and prepared a dataset (diabetes dataset)
2. Trained a Random Forest regression model
3. Registered the model in Azure ML workspace
4. Created a scoring script for inference
5. Deployed the model to Azure Container Instance (ACI)
6. Tested the deployed service
7. Generated example code for making predictions

**Your deployed model is now accessible via REST API!**

### About the Azure ML Resource:

The resource you created is an **Azure Machine Learning Workspace**. This is correct and exactly what you need! It provides:
- Model registration and versioning
- Environment management
- Deployment capabilities (ACI, AKS, etc.)
- Monitoring and logging

### Next Steps:

1. **Replace the dataset**: Update Step 4 to load your own dataset
2. **Customize the model**: Modify Step 6 to use your preferred algorithm
3. **Scale up**: When ready for production, consider deploying to Azure Kubernetes Service (AKS) instead of ACI
4. **Monitor**: Use Azure ML Studio to monitor your model's performance

### Cost Considerations:

- ACI deployment is charged hourly while running
- Remember to delete the service when not in use to avoid charges
- You can always redeploy using this notebook